# Geometria muon veto CUPID — settori attivi ~40 cm + fori (3D orientabile)

I **~71 settori attivi** del muon veto (sorgente `cad/water_sectors.csv`), con la segmentazione
delle slide 16/17 e i **fori** (`cad/holes.json`):
- **Tetto (Z+)**: griglia ~40 cm (più finemente segmentato, come slide 16) con **foro Ø1.5 m** intagliato (passaggio criostato)
- **Fondo (Z−)**: strisce ~40 cm con **foro Ø1.5 m** (cerchio nero, gestito dinamicamente nel modello)
- **Pareti (X±, Y±)**: strisce verticali ~40 cm (fibre lungo l'altezza) con una **porta rettangolare** di accesso/passaggio cavi ciascuna
- Pareti da **1 cm** tra i settori (spezzano la traccia → meno luce per settore)

Colorati per faccia. Ruota/zooma col mouse. Ad ogni settore si assegna l'efficienza del prototipo
(`mc_response_<geo>`) campionata sul cammino nel settore.

In [25]:
import os, csv, json
import numpy as np
import plotly.graph_objects as go

CAD="/Users/benussi/Testbeam2026_WC_unified/cad"
# settori attivi ~40 cm (da water_sectors.csv) + fori (holes.json)
rows=list(csv.DictReader(open(os.path.join(CAD,"water_sectors.csv"))))
for r in rows:
    for k in r:
        if k != "face": r[k]=float(r[k])
mods=[dict(xmin=r["xmin"],ymin=r["ymin"],zmin=r["zmin"],
           xmax=r["xmax"],ymax=r["ymax"],zmax=r["zmax"],
           cx=r["cx"],cy=r["cy"],cz=r["cz"],face=r["face"]) for r in rows]
n=len(mods)
HOLES=json.load(open(os.path.join(CAD,"holes.json")))

WCOL={"Xp":"#d62728","Xm":"#ff9896","Yp":"#1f77b4","Ym":"#aec7e8","Zp":"#2ca02c","Zm":"#98df8a"}
WLAB={"Xp":"X+ ","Xm":"X- ","Yp":"Y+ ","Ym":"Y- ","Zp":"Z+ (tetto)","Zm":"Z- (fondo)"}
def wall_of(m): return m["face"]
from collections import Counter
print(f"settori attivi: {n}")
print("per faccia:", dict(Counter(m['face'] for m in mods)))
print(f"fori: tetto Oe{2*HOLES['top_circle']['r']/1000:.1f}m, fondo Oe{2*HOLES['bot_circle']['r']/1000:.1f}m, "
      f"{len(HOLES['wall_ports'])} porte sulle pareti")

settori attivi: 74
per faccia: {'Xp': 9, 'Yp': 9, 'Xm': 9, 'Zp': 29, 'Zm': 9, 'Ym': 9}
fori: tetto Oe0.4m, fondo Oe1.5m, 4 porte sulle pareti


In [26]:
def box_mesh(m,color):
    x0,y0,z0,x1,y1,z1=m["xmin"],m["ymin"],m["zmin"],m["xmax"],m["ymax"],m["zmax"]
    X=[x0,x1,x1,x0,x0,x1,x1,x0];Y=[y0,y0,y1,y1,y0,y0,y1,y1];Z=[z0,z0,z0,z0,z1,z1,z1,z1]
    I=[0,0,0,0,4,4,1,1,2,2,3,0];J=[1,2,4,3,5,7,5,6,6,7,7,4];K=[2,3,5,7,7,4,6,2,7,3,4,1]
    return go.Mesh3d(x=X,y=Y,z=Z,i=I,j=J,k=K,color=color,opacity=0.55,flatshading=True,hoverinfo="skip",showlegend=False)
def box_edges(m,color):
    x0,y0,z0,x1,y1,z1=m["xmin"],m["ymin"],m["zmin"],m["xmax"],m["ymax"],m["zmax"]
    V=[(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),(x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
    E=[(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    xs=[];ys=[];zs=[]
    for a,b in E: xs+=[V[a][0],V[b][0],None];ys+=[V[a][1],V[b][1],None];zs+=[V[a][2],V[b][2],None]
    return go.Scatter3d(x=xs,y=ys,z=zs,mode="lines",line=dict(color=color,width=2),hoverinfo="skip",showlegend=False)
def circle(c,r,z,name=None):
    ph=np.linspace(0,2*np.pi,120)
    return go.Scatter3d(x=c[0]+r*np.cos(ph),y=c[1]+r*np.sin(ph),z=np.full_like(ph,z),mode="lines",
                        line=dict(color="black",width=6),name=name or "",showlegend=bool(name),hoverinfo="skip")
def port_rect(p):
    wax=p["wax"]; nrm=p["nrm"]
    ww=[p["wlo"],p["whi"],p["whi"],p["wlo"],p["wlo"]]; zz=[p["zlo"],p["zlo"],p["zhi"],p["zhi"],p["zlo"]]
    xs=[];ys=[];zs=[]
    for w,z in zip(ww,zz):
        pt=[0.0,0.0,0.0]; pt[wax]=w; pt[nrm]=p["ncoord"]; pt[2]=z
        xs.append(pt[0]);ys.append(pt[1]);zs.append(pt[2])
    return go.Scatter3d(x=xs,y=ys,z=zs,mode="lines",line=dict(color="black",width=5),hoverinfo="skip",showlegend=False)

fig=go.Figure()
for m in mods:
    c=WCOL.get(wall_of(m),"#888")
    fig.add_trace(box_mesh(m,c)); fig.add_trace(box_edges(m,c))
# fori: cerchi tetto/fondo + porte pareti
fig.add_trace(circle(HOLES["top_circle"]["c"],HOLES["top_circle"]["r"],HOLES["top_circle"]["z"],name="foro Ø1.5 m"))
fig.add_trace(circle(HOLES["bot_circle"]["c"],HOLES["bot_circle"]["r"],HOLES["bot_circle"]["z"]))
for p in HOLES["wall_ports"]: fig.add_trace(port_rect(p))
# legenda per faccia
for w,c in WCOL.items():
    fig.add_trace(go.Scatter3d(x=[None],y=[None],z=[None],mode="markers",marker=dict(size=8,color=c),name=WLAB[w]))
fig.update_layout(title=f"CUPID muon veto — {n} settori attivi ~40 cm + fori (slide 16/17)",
                  scene=dict(xaxis_title="X (mm)",yaxis_title="Y (mm)",zaxis_title="Z (mm)",aspectmode="data"),
                  width=950,height=800,legend=dict(title="faccia / fori"))
fig.show()

In [27]:
# sovrappone alcune tracce di muoni cosmici che attraversano >=2 moduli
import sys; sys.path.insert(0,"/Users/benussi/Testbeam2026_WC_unified/Analysis_script_background")
import muon_multiplicity as MM
rng=np.random.default_rng(3); o,d=MM.gen_muons(4000,rng)
tracks=[]
for i in range(len(o)):
    L=MM.paths_chunk(o[i:i+1],d[i:i+1])[0]
    if (L>0.5).sum()>=2:
        tmax=(MM.Zr[0]-o[i,2])/d[i,2]
        tracks.append((o[i],o[i]+d[i]*tmax))
    if len(tracks)>=15: break
for p0,p1 in tracks:
    fig.add_trace(go.Scatter3d(x=[p0[0],p1[0]],y=[p0[1],p1[1]],z=[p0[2],p1[2]],
                  mode="lines",line=dict(color="black",width=4),hoverinfo="skip",showlegend=False))
fig.update_layout(title=f"CUPID muon veto — {n} settori + tracce muoniche di esempio")
fig.show(); print(f"{len(tracks)} tracce disegnate")

15 tracce disegnate
